In [1]:
import pandas as pd
import numpy as np

-----------------------------------------
part 1
---

In [4]:
df_events = pd.read_csv('../02-event-cleaning-pandas/events_cleaned.csv')
df_users = pd.read_csv('./users.csv')
df_metrics = pd.read_csv('./system_metrics.csv')

**2**
exploring

In [30]:
df_events.head()

,event_id,timestamp,event_type,user_id,status,latency_ms
0,928,2026-01-15 21:39:53,checkout,user_123,error,316.0
1,631,2026-01-15 19:35:45,purchase,user_253,ok,1652.0
2,683,2026-01-15 01:07:47,purchase,user_154,error,1543.0
3,515,2026-01-15 10:26:46,view,user_257,error,733.0
4,366,2026-01-15 20:11:32,checkout,user_925,error,588.0


In [6]:
df_metrics.head()

,timestamp,cpu_usage,memory_usage,error_rate,events_per_second
0,2026-01-15 00:00:00,61.56,30.57,0.2816,1491
1,2026-01-15 01:00:00,21.63,66.04,0.1575,1295
2,2026-01-15 02:00:00,37.88,64.09,0.1985,1196
3,2026-01-15 03:00:00,34.51,56.85,0.1202,1334
4,2026-01-15 04:00:00,67.87,43.34,0.2522,1776


In [8]:
df_users.head()

,user_id,name,region,signup_date,account_type,email
0,user_100,User 2824,APAC,2025-08-11,premium,user0@example.com
1,user_101,User 1409,US-East,2026-01-08,free,user1@example.com
2,user_102,User 5506,US-East,2025-06-18,standard,user2@example.com
3,user_103,User 5012,US-East,2026-02-03,free,user3@example.com
4,user_104,User 4657,APAC,2025-12-15,free,user4@example.com


common ground
df_users **X** df_events
- user_id

In [11]:
df_users.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   user_id       200 non-null    str  
 1   name          200 non-null    str  
 2   region        200 non-null    str  
 3   signup_date   200 non-null    str  
 4   account_type  200 non-null    str  
 5   email         200 non-null    str  
dtypes: str(6)
memory usage: 9.5 KB


In [12]:
df_metrics.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   timestamp          100 non-null    str    
 1   cpu_usage          100 non-null    float64
 2   memory_usage       100 non-null    float64
 3   error_rate         100 non-null    float64
 4   events_per_second  100 non-null    int64  
dtypes: float64(3), int64(1), str(1)
memory usage: 4.0 KB


In [13]:
df_events.info()

<class 'pandas.DataFrame'>
RangeIndex: 955 entries, 0 to 954
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   event_id    955 non-null    int64  
 1   timestamp   955 non-null    str    
 2   event_type  955 non-null    str    
 3   user_id     936 non-null    str    
 4   status      955 non-null    str    
 5   latency_ms  955 non-null    float64
dtypes: float64(1), int64(1), str(4)
memory usage: 44.9 KB


In [14]:
df_users.describe()

,user_id,name,region,signup_date,account_type,email
count,200,200,200,200,200,200
unique,200,198,5,136,3,200
top,user_100,User 3504,US-West,2025-11-18,free,user0@example.com
freq,1,2,47,4,87,1


In [15]:
df_events.describe()

,event_id,latency_ms
count,955.000000,955.000000
mean,499.531937,1012.571687
std,288.034843,541.218537
min,1.000000,50.000000
25%,252.500000,580.500000
50%,500.000000,1015.278835
75%,746.500000,1473.500000
max,1000.000000,2000.000000


In [16]:
df_metrics.describe()

,cpu_usage,memory_usage,error_rate,events_per_second
count,100.000000,100.000000,100.000000,100.000000
mean,51.171800,54.444100,0.204623,1502.620000
std,19.175033,14.656812,0.065260,246.724923
min,20.420000,30.030000,0.081700,1037.000000
25%,34.735000,41.495000,0.158325,1294.750000
50%,51.340000,53.490000,0.203000,1517.000000
75%,66.197500,68.055000,0.254800,1704.250000
max,84.840000,79.960000,0.341000,2038.000000


**find uniq users**

In [22]:
len(df_users) - df_users['user_id'].nunique()

0

In [26]:
df_events['user_id'].nunique()

554

In [27]:
df_users.isnull().sum()

user_id         0
name            0
region          0
signup_date     0
account_type    0
email           0
dtype: int64

**events**

In [24]:
df_events['event_type'].unique()

<StringArray>
['checkout', 'purchase', 'view', 'login', 'unknown']
Length: 5, dtype: str

In [28]:
df_events.isnull().sum()

event_id       0
timestamp      0
event_type     0
user_id       19
status         0
latency_ms     0
dtype: int64

**matrix**

In [29]:
df_metrics.isnull().sum()

timestamp            0
cpu_usage            0
memory_usage         0
error_rate           0
events_per_second    0
dtype: int64

------------------------------------------------------------
Part 2: Understanding Join Relationships
---

**part 5**

In [45]:
events_users = set(df_events['user_id'].unique())
user_ids = set(df_users['user_id'].unique())
missing_users = events_users - user_ids

len(missing_users)

444

In [43]:
events_per_user = df_events.groupby('user_id').size()
print(f"  Events per user: {events_per_user.min()} to {events_per_user.max()}")
print(f"  Average: {events_per_user.mean():.1f}")


  Events per user: 1 to 7
  Average: 1.7


**part 6**

In [46]:
df_events['timestamp'] = pd.to_datetime(df_events['timestamp'])
df_metrics['timestamp'] = pd.to_datetime(df_metrics['timestamp'])

df_events.info()

<class 'pandas.DataFrame'>
RangeIndex: 955 entries, 0 to 954
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   event_id    955 non-null    int64         
 1   timestamp   955 non-null    datetime64[us]
 2   event_type  955 non-null    str           
 3   user_id     936 non-null    str           
 4   status      955 non-null    str           
 5   latency_ms  955 non-null    float64       
dtypes: datetime64[us](1), float64(1), int64(1), str(3)
memory usage: 44.9 KB


In [48]:
df_metrics.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   timestamp          100 non-null    datetime64[us]
 1   cpu_usage          100 non-null    float64       
 2   memory_usage       100 non-null    float64       
 3   error_rate         100 non-null    float64       
 4   events_per_second  100 non-null    int64         
dtypes: datetime64[us](1), float64(3), int64(1)
memory usage: 4.0 KB


In [53]:
print(f"  Events timestamp range: {df_events['timestamp'].min()} to {df_events['timestamp'].max()}")
print(f"  Metrics timestamp range: {df_metrics['timestamp'].min()} to {df_metrics['timestamp'].max()}")
print(f"  Events granularity: per-event")
print(f"  Metrics granularity: {(df_metrics['timestamp'].iloc[1] - df_metrics['timestamp'].iloc[0]).total_seconds() / 3600} hours")


  Events timestamp range: 2026-01-15 00:00:53 to 2050-01-15 23:34:56
  Metrics timestamp range: 2026-01-15 00:00:00 to 2026-01-19 03:00:00
  Events granularity: per-event
  Metrics granularity: 1.0 hours


-------------------------------------------
Part 4: Feature Engineering
----